# 03. Сравнение моделей и оценка качества эмбеддингов

**Цель:** Сравнить разные подходы к генерации рекомендаций и обосновать выбор Sentence-Transformers + ChromaDB.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")

df = pd.read_csv("high_popularity_spotify_data.csv")
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (1686, 20)


,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,track_popularity,track_album_name,track_id,track_name,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,playlist_id
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",100,Die With A Smile,2plbrEY59IikOBgBGLjaoe,Die With A Smile,10FLjwfpbxLmW8c25Xyc2N,0,6,251668,0.3080,2plbrEY59IikOBgBGLjaoe,mainstream,37i9dQZF1DXcBWIGoYBM5M
1,0.507,104.978,0.747,pop,-10.171,0.117,0.438,Billie Eilish,97,HIT ME HARD AND SOFT,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,7aJuG4TFXa2hmE4z1yxc3n,1,2,210373,0.2000,6dOtVTDdiauQNBQEDOtlAB,mainstream,37i9dQZF1DXcBWIGoYBM5M
2,0.808,108.548,0.554,pop,-4.169,0.159,0.372,Gracie Abrams,93,The Secret of Us (Deluxe),7ne4VBA60CxGM75vw0EYad,That’s So True,0hBRqPYPXhr1RkTDG3n4Mk,1,1,166300,0.2140,7ne4VBA60CxGM75vw0EYad,mainstream,37i9dQZF1DXcBWIGoYBM5M
3,0.910,112.966,0.670,pop,-4.070,0.304,0.786,Sabrina Carpenter,81,Short n' Sweet,1d7Ptw3qYcfpdLNL5REhtJ,Taste,4B4Elma4nNDUyl6D5PvQkj,0,0,157280,0.0939,1d7Ptw3qYcfpdLNL5REhtJ,mainstream,37i9dQZF1DXcBWIGoYBM5M
4,0.783,149.027,0.777,pop,-4.477,0.355,0.939,"ROSÉ, Bruno Mars",98,APT.,5vNRhkKd0yEAg8suGBpjeY,APT.,2IYQwwgxgOIn7t3iF6ufFD,0,0,169917,0.0283,5vNRhkKd0yEAg8suGBpjeY,mainstream,37i9dQZF1DXcBWIGoYBM5M


In [2]:
# Базовая предобработка
df = df.drop_duplicates(subset=['track_name', 'track_artist']).reset_index(drop=True)
print(f"After deduplication: {df.shape}")

After deduplication: (1420, 20)


In [3]:
from sentence_transformers import SentenceTransformer
import time
from sklearn.metrics.pairwise import cosine_similarity

# === Модели для сравнения ===
models_to_test = {
    "all-MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",      # Быстрый и хороший
    "paraphrase-multilingual-MiniLM-L12-v2": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "distiluse-base-multilingual-cased-v1": "sentence-transformers/distiluse-base-multilingual-cased-v1"
}

results = []

for name, model_name in models_to_test.items():
    print(f"\nЗагрузка модели: {name}")
    start = time.time()

    model = SentenceTransformer(model_name)
    load_time = time.time() - start

    # Создаём текстовые представления
    texts = []
    for _, row in df.head(500).iterrows():  # Берём подвыборку для скорости
        text = f"Song: {row['track_name']} | Artist: {row['track_artist']} | Genre: {row.get('playlist_genre', '')}"
        texts.append(text)

    # Кодирование
    start = time.time()
    embeddings = model.encode(texts, show_progress_bar=False)
    encode_time = time.time() - start

    results.append({
        "Model": name,
        "Load Time (s)": round(load_time, 2),
        "Encode Time (500 samples)": round(encode_time, 2),
        "Embedding Dim": embeddings.shape[1],
        "Memory Usage (MB)": round(embeddings.nbytes / (1024*1024), 2)
    })

comparison_df = pd.DataFrame(results)
display(comparison_df)


Загрузка модели: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

,Model,Load Time (s),Encode Time (500 samples),Embedding Dim,Memory Usage (MB)
0,all-MiniLM-L6-v2,6.38,6.52,384,0.73
1,paraphrase-multilingual-MiniLM-L12-v2,15.50,14.48,384,0.73
2,distiluse-base-multilingual-cased-v1,17.47,21.99,512,0.98


In [4]:
# Примеры тестовых запросов
test_queries = [
    "rainy coffee shop morning chill",
    "energetic morning workout",
    "romantic evening date",
    "focused deep work coding session",
    "happy summer party"
]

# Используем лучшую модель
best_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
query_embeddings = best_model.encode(test_queries)

# Получаем эмбеддинги треков
track_texts = [f"{row['track_name']} by {row['track_artist']}" for _, row in df.iterrows()]
track_embeddings = best_model.encode(track_texts[:2000])  # ограничиваем для скорости

print("Готово к оценке качества рекомендаций")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Готово к оценке качества рекомендаций


In [5]:
def get_recommendations_for_query(query_idx, top_k=5):
    query_emb = query_embeddings[query_idx]
    similarities = cosine_similarity([query_emb], track_embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]

    recs = []
    for idx in top_indices:
        recs.append({
            "rank": len(recs)+1,
            "song": df.iloc[idx]['track_name'],
            "artist": df.iloc[idx]['track_artist'],
            "score": round(float(similarities[idx]), 4)
        })
    return recs

# Пример
for i, query in enumerate(test_queries):
    print(f"\n{'='*60}")
    print(f"Запрос: {query}")
    print(f"{'='*60}")
    recs = get_recommendations_for_query(i, top_k=5)
    for rec in recs:
        print(f"{rec['rank']}. {rec['song']} — {rec['artist']} (score: {rec['score']})")


Запрос: rainy coffee shop morning chill
1. Sweater Weather — The Neighbourhood (score: 0.4682)
2. Eres — Café Tacvba (score: 0.3885)
3. No Rain — Blind Melon (score: 0.359)
4. Good Morning — Kanye West (score: 0.3564)
5. Break from Toronto — PARTYNEXTDOOR (score: 0.3093)

Запрос: energetic morning workout
1. Good Morning — Kanye West (score: 0.4041)
2. Wake Me up When September Ends — Green Day (score: 0.3117)
3. Circadian Rhythm — Drake (score: 0.3064)
4. Stereo Hearts (feat. Adam Levine) — Gym Class Heroes, Adam Levine (score: 0.3056)
5. Wake Me Up — Avicii (score: 0.303)

Запрос: romantic evening date
1. Romantic Homicide — d4vd (score: 0.4029)
2. Sex, Drugs, Etc. — Beach Weather (score: 0.3673)
3. First Date — blink-182 (score: 0.3648)
4. Heartbreak Anniversary — Giveon (score: 0.353)
5. Lover — Taylor Swift (score: 0.3268)

Запрос: focused deep work coding session
1. Thinking out Loud — Ed Sheeran (score: 0.2506)
2. Work Song — Hozier (score: 0.2476)
3. Wide Awake — Katy Perry (s

In [6]:
print("""
## Итоговые выводы

Выбрана модель: **all-MiniLM-L6-v2**

**Преимущества:**
- Отличное соотношение качество/скорость
- Маленький размер (≈80MB)
- Хорошо работает с короткими текстами (названия + артист)
- Быстрое кодирование на CPU
- Хорошие результаты на семантическом поиске музыки

**Альтернативы:**
- Более тяжёлые модели дают небольшой прирост качества, но сильно теряют в скорости и потреблении памяти.
""")


## Итоговые выводы

Выбрана модель: **all-MiniLM-L6-v2**

**Преимущества:**
- Отличное соотношение качество/скорость
- Маленький размер (≈80MB)
- Хорошо работает с короткими текстами (названия + артист)
- Быстрое кодирование на CPU
- Хорошие результаты на семантическом поиске музыки

**Альтернативы:**
- Более тяжёлые модели дают небольшой прирост качества, но сильно теряют в скорости и потреблении памяти.

